In [7]:
import pandas as pd
from scipy import stats
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.formula.api import ols
import statsmodels.api as sm

### Cleaning 

In [8]:
df = pd.read_excel('augmented_survey.xlsx')

df['gender']         = df['Q1. What is your gender?']
df['allowance']      = df['Q2. What is your approximate monthly allowance / pocket money?']
df['year']           = df['Q3. Which year of college are you in?']
df['venue']          = df['Q4. What type of venue did you go to?']
df['day']            = df['Q5. What day was the outing?']
df['intiated_plan']  = df['Q6. Who initiated / planned the outing?']
df['occasion']       = df['Q7. What was the occasion?']
df['group_size']     = df['Q8. How many people were in your group, including yourself?']
df['travel_dist']    = df['Q9. How far did you travel to reach the venue?']
df['spending']       = df['Q10. Approximately how much did YOU personally spend on this outing? (₹) (Include food, travel, entry, shopping — everything)']
df['rough_budget']   = df['Q11. Did you have a rough budget in mind before going?']
df['exceed_budget']  = df['Q12. Did your actual spending exceed that budget? (Answer only if Q11 = Yes)']
df['spend_reason']   = df['Q13. What was the biggest reason you spent what you did? (Pick one)']
df['outing_freq']    = df['Q14. How often do you go on social outings per month?']
df['peer_influence'] = df["Q15. On a scale of 1–5, how much do your friends' choices influence where you go and what you spend? (1 = Not at all, 5 = Completely driven by friends)"]
df['discount']       = df['Q16. Did you use any discount or offer? (Zomato, Swiggy Dineout, student discount, etc.)']

df = df.drop(columns={'Q1. What is your gender?',
       'Q2. What is your approximate monthly allowance / pocket money?',
       'Q3. Which year of college are you in?',
       'Q4. What type of venue did you go to?', 'Q5. What day was the outing?',
       'Q6. Who initiated / planned the outing?', 'Q7. What was the occasion?',
       'Q8. How many people were in your group, including yourself?',
       'Q9. How far did you travel to reach the venue?',
       'Q10. Approximately how much did YOU personally spend on this outing? (₹) (Include food, travel, entry, shopping — everything)',
       'Q11. Did you have a rough budget in mind before going?',
       'Q12. Did your actual spending exceed that budget? (Answer only if Q11 = Yes)',
       'Q13. What was the biggest reason you spent what you did? (Pick one)',
       'Q14. How often do you go on social outings per month?',
       "Q15. On a scale of 1–5, how much do your friends' choices influence where you go and what you spend? (1 = Not at all, 5 = Completely driven by friends)",
       'Q16. Did you use any discount or offer? (Zomato, Swiggy Dineout, student discount, etc.)'})
df

,Timestamp,"If 'Other' for Q4, please specify the venue type.","If 'Other' for Q7, please specify the occasion.",Email Address,gender,allowance,year,venue,day,intiated_plan,occasion,group_size,travel_dist,spending,rough_budget,exceed_budget,spend_reason,outing_freq,peer_influence,discount
0,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-03-22 09:51:52.551,NaN,NaN,alisha.s.rao@gmail.com,Female,"Less than ₹3,000",2nd Year,Café / Coffee Shop,Weekday (Mon–Thu),A friend did,Just hanging out (no specific reason),3–4 people,Less than 2 km,₹200 – ₹500,No,"Yes, I spent more than I planned",I was in a good mood,2–3 times,4.0,No
2,2026-03-28 10:53:14.288,NaN,NaN,snigdhamishra2006@gmail.com,Female,"₹10,001 – ₹15,000",2nd Year,Movie Theatre,Weekday (Mon–Thu),I did,Just hanging out (no specific reason),5–7 people,2–5 km,"₹501 – ₹1,000",Yes,"Yes, I spent more than I planned",I was in a good mood,4–5 times,3.0,Yes
3,2026-03-28 10:58:04.500,NaN,NaN,zoyaabhatkar@gmail.com,Female,"₹3,000 – ₹6,000",2nd Year,Café / Coffee Shop,Weekend (Sat–Sun),It was a group decision,Just hanging out (no specific reason),5–7 people,6–10 km,"₹501 – ₹1,000",Yes,"No, I stayed within my budget",The venue/place itself was expensive,2–3 times,3.0,No
4,2026-03-28 10:59:00.192,NaN,NaN,ayaanalikhunt@gmail.com,Male,"₹6,001 – ₹10,000",1st Year,Pub / Bar / Lounge,Weekday (Mon–Thu),I did,After exams / stress relief,2 people,Less than 2 km,"₹501 – ₹1,000",Yes,"No, I stayed within my budget",I just didn't track / wasn't paying attention,2–3 times,3.0,No
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
416,2026-04-06 08:13:23.821,NaN,NaN,stewarttracey@example.org,Female,"Less than ₹3,000",1st Year,Mall / Shopping,Weekday (Mon–Thu),A friend did,Just hanging out (no specific reason),3–4 people,More than 10 km,"₹501 – ₹1,000",No,"No, I stayed within my budget",My friends were spending so I did too (peer in...,4–5 times,3.0,Yes
417,2026-03-31 00:27:50.132,NaN,NaN,gabriellamartinez@example.com,Female,"Less than ₹3,000",2nd Year,Park / Outdoor space,Weekday (Mon–Thu),A friend did,Just hanging out (no specific reason),5–7 people,More than 10 km,₹200 – ₹500,Yes,NaN,NaN,2–3 times,4.0,Yes
418,2026-03-31 03:07:16.593,NaN,NaN,margaret20@example.org,Female,"Less than ₹3,000",1st Year,Café / Coffee Shop,Weekend (Sat–Sun),"It was spontaneous, no one planned it",Just hanging out (no specific reason),8 or more,2–5 km,"₹1,001 – ₹2,000",No,NaN,The venue/place itself was expensive,Once or less,5.0,Yes
419,NaT,NaN,NaN,hortonbarbara@example.com,Female,"Less than ₹3,000",2nd Year,Restaurant / Dhaba,Weekday (Mon–Thu),It was a group decision,Just hanging out (no specific reason),3–4 people,2–5 km,"₹1,001 – ₹2,000",Yes,I spent exactly what I planned,It was a special occasion so I didn't hold back,2–3 times,3.0,No


In [9]:
spend_map = {
    'Less than ₹200': 100,
    '₹200 – ₹500': 350,
    '₹501 – ₹1,000': 750,
    '₹1,001 – ₹2,000': 1500,
    'More than ₹2,000': 2500
}
allowance_map = {
    'Less than ₹3,000': 1500,
    '₹3,000 – ₹6,000': 4500,
    '₹6,001 – ₹10,000': 8000,
    '₹10,001 – ₹15,000': 12500,
    'More than ₹15,000': 17500
}

df['spending']  = df['spending'].map(spend_map)
df['allowance'] = df['allowance'].map(allowance_map)

In [12]:
df_real = df.iloc[:141]

In [ ]:
df_real

# Hypothesis Testing 

### Q1 Do male and female students spend significantly differently? (independent samples t-test)


In [15]:
male   = df_real[df_real['gender'] == 'Male']['spending'].dropna()
female = df_real[df_real['gender'] == 'Female']['spending'].dropna()

t_stat, p_value = stats.ttest_ind(male, female)
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")
print("Significant difference" if p_value < 0.05 else "No significant difference")

T-statistic: 0.3575, P-value: 0.7213
No significant difference


### Q2 Do students who had a budget (Q11=Yes) spend less than those who didn't? (t-test)

In [16]:
budget_yes = df_real[df_real['rough_budget'] == 'Yes']['spending'].dropna()
budget_no  = df_real[df_real['rough_budget'] == 'No']['spending'].dropna()

t_stat, p_value = stats.ttest_ind(budget_yes, budget_no, alternative='less')
print(f"T-statistic: {t_stat:.4f}, P-value: {p_value:.4f}")
print("Budget group spends significantly less" if p_value < 0.05 else "No significant difference")

T-statistic: -0.7992, P-value: 0.2128
No significant difference


### Is the proportion of discount users (Q16) different between genders? (z-test for proportions)

In [17]:
from statsmodels.stats.proportion import proportions_ztest

male_data   = df_real[df_real['gender'] == 'Male']['discount'].dropna()
female_data = df_real[df_real['gender'] == 'Female']['discount'].dropna()

count = [(male_data == 'Yes').sum(),   (female_data == 'Yes').sum()]
nobs  = [len(male_data),               len(female_data)]

z_stat, p_value = proportions_ztest(count, nobs)
print(f"Z-statistic: {z_stat:.4f}, P-value: {p_value:.4f}")
print("Significant difference in discount usage" if p_value < 0.05 else "No significant difference")

Z-statistic: 0.1551, P-value: 0.8767
No significant difference


# Conclusion
 ## Q1 — Gender vs Spending
## Male and female students show no statistically significant difference in outing expenditure (p = 0.72), suggesting that gender does not play a meaningful role in determining how much college students spend on social outings.
 ## Q2 — Budget vs Spending
 ## Students who set a rough budget beforehand do not spend significantly less than those who didn't (p = 0.21), indicating that merely having a budget in mind does not effectively constrain actual spending behaviour.
## Q3 — Gender vs Discount Usage
## The proportion of discount users is not significantly different between male and female students (p = 0.88), implying that discount-seeking behaviour during outings is equally prevalent across genders.